# 1. Basic Voice Pipeline — STT + LLM + TTS

**Agenda slot: 0–20 min**

This is the naive pipeline most tutorials show: wait for each stage to fully
finish before starting the next one. It works, and it's the easiest way to
understand the moving parts — but it's slow. Notebook 2 shows how production
systems fix that.

Full agenda: [`../voice-agent.md`](../voice-agent.md)

## 0. Why Voice is the Next UI

The interface layer keeps shifting:

- **Desktop → GUI**
- **Mobile → Touch**
- **ChatGPT → Text**
- **AI Agents → Voice**

Where this shows up already: customer support, restaurant ordering, medical
assistants, personal assistants, coding assistants.

> **Question for the room:** why is voice harder than chat?

## 1. The Naive Pipeline

```
User
 ↓
Speech-to-Text   (Whisper)
 ↓
LLM              (GPT)
 ↓
Tool Calls       (Database/API)
 ↓
LLM
 ↓
Text-to-Speech
 ↓
Audio
```

Drawbacks of this version — because we wait for each stage to fully finish:

- Multiple network round-trips
- Noticeable end-to-end latency
- No handling for interruptions — the user can't barge in
- Nothing streams — you wait for each stage to fully finish
- The result feels robotic, not conversational

Let's run it live — start to finish, timing every step.

### Mic utilities

Two helpers used throughout: `record_audio()` captures from the mic to a WAV
file, `play_audio()` plays a WAV file through the speakers. Grant microphone
access to Jupyter when the OS prompts.

In [ ]:
import sounddevice as sd
import soundfile as sf


def record_audio(filename="input.wav", duration=5, samplerate=16000):
    print(f"Recording for {duration}s...")
    audio = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1)
    sd.wait()
    sf.write(filename, audio, samplerate)
    print(f"Saved to {filename}")
    return filename


def play_audio(filename):
    audio, samplerate = sf.read(filename)
    sd.play(audio, samplerate)
    sd.wait()

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import time

load_dotenv()
client = OpenAI()

Record a real question from the mic — grab a volunteer from the room.

In [ ]:
record_audio("sample_question.wav", duration=5)

In [ ]:
t0 = time.time()
transcript = client.audio.transcriptions.create(model="whisper-1", file=open("sample_question.wav", "rb"), language="en").text
transcript

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise voice assistant. Always reply in English."},
    {"role": "user", "content": transcript},
]
answer = client.chat.completions.create(model="gpt-4o-mini", messages=messages).choices[0].message.content
answer

In [ ]:
with client.audio.speech.with_streaming_response.create(model="tts-1", voice="alloy", input=answer, response_format="wav") as response:
    response.stream_to_file("sample_answer.wav")
print(f"Total pipeline time: {time.time() - t0:.1f}s")
play_audio("sample_answer.wav")

Three separate API calls, each waited on fully, just to answer one question.
Notice the total time printed above — typically a couple of seconds.

**Next up:** `02_optimized_streaming.ipynb` — how production systems shrink
this down to under a second without changing the fundamental STT → LLM → TTS
shape.